# VinAI Translate baseline trên Kaggle

Notebook dùng cùng code, config, dataset và seed 67 với Colab. Trước khi chạy: bật **GPU** và **Internet**, sau đó upload `NLP_colab.zip` (v2) hoặc `NLP_v3_parent.zip` (audit/freeze v3) thành Kaggle Input. Kaggle có thể tự giải nén ZIP; notebook hỗ trợ cả hai dạng và cố định một GPU để effective batch không đổi.

## 1. Kiểm tra GPU và môi trường

Thành công khi CUDA khả dụng và cell in đúng tên GPU.

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # giữ effective batch giống Colab/nhóm
import subprocess, sys
subprocess.run(["nvidia-smi"], check=False)
import torch
print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Hãy bật GPU trong Kaggle Notebook settings.")
print("GPU dùng cho baseline:", torch.cuda.get_device_name(0))
print("Số GPU nhìn thấy bởi tiến trình:", torch.cuda.device_count())

## 2. Nhận diện project từ Kaggle Input

Kaggle có thể tự giải nén ZIP khi tạo Dataset. Cell này hỗ trợ cả project đã giải nén và file `.zip`, sau đó sao chép/giải nén sang `/kaggle/working`. Chỉ đặt `INPUT_NAME_HINT` khi notebook có nhiều Input.

In [ ]:
from pathlib import Path, PurePosixPath
import shutil, tempfile, zipfile

INPUT_NAME_HINT = ""  # ví dụ: nlp-v3-parent; để trống nếu chỉ có một project
input_root = Path("/kaggle/input")
if not input_root.is_dir():
    raise FileNotFoundError("Không tìm thấy /kaggle/input; hãy Add Input trước.")

def is_project_root(path):
    return ((path / "requirements-training.txt").is_file()
            and (path / "src/vinai_baseline/cli.py").is_file()
            and (path / "configs/vinai").is_dir())

def find_project_roots(root):
    roots = [root] if is_project_root(root) else []
    roots += [p.parent for p in root.rglob("requirements-training.txt")
              if p.parent != root and is_project_root(p.parent)]
    return sorted(set(roots), key=lambda p: (len(p.parts), str(p)))

def normalized(value):
    return "".join(ch for ch in str(value).casefold() if ch.isalnum())

hint = normalized(INPUT_NAME_HINT)
def matches_hint(path):
    return not hint or hint in normalized(path)

direct_candidates = [p for p in find_project_roots(input_root) if matches_hint(p)]
zip_candidates = [p for p in sorted(input_root.rglob("*.zip")) if matches_hint(p)]
working_root = Path(tempfile.mkdtemp(prefix="nlp_kaggle_", dir="/kaggle/working"))

if len(direct_candidates) == 1:
    source_project = direct_candidates[0]
    PROJECT_DIR = working_root / "project"
    shutil.copytree(source_project, PROJECT_DIR)
    input_mode = "Kaggle Dataset đã giải nén"
elif not direct_candidates and len(zip_candidates) == 1:
    zip_path = zip_candidates[0]
    extract_dir = working_root / "extracted"
    extract_dir.mkdir()
    extract_root = extract_dir.resolve()
    with zipfile.ZipFile(zip_path) as archive:
        corrupt = archive.testzip()
        if corrupt:
            raise ValueError(f"ZIP bị hỏng tại entry: {corrupt}")
        for member in archive.infolist():
            normalized_name = member.filename.replace("\\", "/")
            relative = PurePosixPath(normalized_name)
            if relative.is_absolute() or ".." in relative.parts or (relative.parts and ":" in relative.parts[0]):
                raise ValueError(f"ZIP chứa đường dẫn không an toàn: {member.filename}")
            if not relative.parts:
                continue
            destination = extract_dir.joinpath(*relative.parts)
            if not destination.resolve().is_relative_to(extract_root):
                raise ValueError(f"ZIP chứa đường dẫn không an toàn: {member.filename}")
            if member.is_dir() or normalized_name.endswith("/"):
                destination.mkdir(parents=True, exist_ok=True)
            else:
                destination.parent.mkdir(parents=True, exist_ok=True)
                with archive.open(member) as source, destination.open("wb") as target:
                    shutil.copyfileobj(source, target)
    extracted_projects = find_project_roots(extract_dir)
    if len(extracted_projects) != 1:
        raise ValueError(f"ZIP phải chứa đúng một project root; tìm thấy: {extracted_projects}")
    PROJECT_DIR = extracted_projects[0]
    input_mode = f"ZIP: {zip_path}"
else:
    all_projects = [str(p) for p in find_project_roots(input_root)]
    all_zips = [str(p) for p in sorted(input_root.rglob("*.zip"))]
    top_level = [str(p) for p in sorted(input_root.iterdir())]
    raise ValueError(
        "Không chọn được đúng một project. "
        f"project={all_projects}, zip={all_zips}, input={top_level}. "
        "Nếu có nhiều Input, hãy đặt INPUT_NAME_HINT theo tên Kaggle Dataset."
    )

os.chdir(PROJECT_DIR)
print("Input mode:", input_mode)
print("Project:", PROJECT_DIR)

## 3. Cài dependencies

Kaggle cần bật Internet để cài đúng stack và tải model public từ Hugging Face. Baseline không dùng TorchAO nên notebook gỡ phiên bản cũ nếu xung đột với PEFT.

In [ ]:
requirement_file = (PROJECT_DIR / "requirements-training-colab.txt"
                    if (PROJECT_DIR / "requirements-training-colab.txt").exists()
                    else PROJECT_DIR / "requirements-training.txt")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(requirement_file)], check=True)
from importlib.metadata import PackageNotFoundError, version
from packaging.version import Version
try:
    torchao_version = version("torchao")
except PackageNotFoundError:
    torchao_version = None
if torchao_version and Version(torchao_version) <= Version("0.16.0"):
    print(f"Gỡ torchao {torchao_version} vì xung đột với PEFT...")
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=True)
print("Dependencies:", requirement_file.name)

## 4. Hugging Face token (không bắt buộc)

Model public vẫn tải được không cần token. Nếu muốn hạn mức ổn định hơn, tạo Kaggle Secret tên `HF_TOKEN`.

In [ ]:
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN")
    if token:
        os.environ["HF_TOKEN"] = token
        print("Đã nạp HF_TOKEN từ Kaggle Secrets.")
except Exception:
    print("Không có HF_TOKEN; tiếp tục với model public.")

## 5. Tạo và duyệt dataset v3 (chỉ dùng với `NLP_v3_parent`)

Parent v3 chưa được phép train. Lần đầu đặt `RUN_V3_SEMANTIC_AUDIT=True` để chấm LaBSE, sau đó xem `semantic_audit.json`. Chỉ khi nhóm đã chốt threshold mới điền `SEMANTIC_THRESHOLD` để freeze và sinh config v3. Với ZIP v2 hoặc frozen v3, giữ nguyên các giá trị mặc định để bỏ qua phần này.

In [ ]:
import json

RUN_V3_SEMANTIC_AUDIT = False
SEMANTIC_THRESHOLD = None  # ví dụ 0.72; chỉ điền sau khi đã duyệt audit
V3_PARENT_DIR = PROJECT_DIR / "artifacts/processed_token448_seed67_v3_parent"
V3_AUDIT_DIR = PROJECT_DIR / "artifacts/processed_semantic_audit_seed67_v3"
V3_FROZEN_DIR = PROJECT_DIR / "artifacts/processed_clean_semantic_seed67_v3"
V3_CONFIG_DIR = PROJECT_DIR / "configs/generated_v3"

if RUN_V3_SEMANTIC_AUDIT or SEMANTIC_THRESHOLD is not None:
    if not V3_PARENT_DIR.joinpath("manifest.json").is_file():
        raise FileNotFoundError(f"Không tìm thấy parent v3: {V3_PARENT_DIR}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    str(PROJECT_DIR / "requirements-semantic.txt")], check=True)

if RUN_V3_SEMANTIC_AUDIT:
    if V3_AUDIT_DIR.exists() and any(V3_AUDIT_DIR.iterdir()):
        raise FileExistsError(f"Audit output đã tồn tại: {V3_AUDIT_DIR}")
    audit_command = [sys.executable, "-u", "-m", "src.data.finalize",
        "--input-dir", str(V3_PARENT_DIR), "--output-dir", str(V3_AUDIT_DIR),
        "--dataset-id", "legal_envi_semantic_audit_seed67_v3",
        "--split-seed", "67", "--minimum-alignment-score", "0.75",
        "--semantic-model", "sentence-transformers/LaBSE",
        "--semantic-model-revision", "836121a", "--device", "cuda"]
    subprocess.run(audit_command, cwd=PROJECT_DIR, check=True)
    print("Semantic audit hoàn thành:", V3_AUDIT_DIR)

audit_report_path = V3_AUDIT_DIR / "semantic_audit.json"
if audit_report_path.is_file():
    audit_report = json.loads(audit_report_path.read_text(encoding="utf-8"))
    for split, values in audit_report["splits"].items():
        print(split, "records=", values["records"],
              "truncated=", f"{values['truncated_rate']:.2f}%",
              "range=", (values["minimum"], values["maximum"]))
    audit_zip = shutil.make_archive("/kaggle/working/semantic_audit_seed67_v3", "zip",
                                    root_dir=V3_AUDIT_DIR)
    print("Hãy tải report/ZIP để nhóm duyệt trước khi freeze:", audit_zip)
elif V3_PARENT_DIR.is_dir():
    print("Đã nhận diện parent v3. Đặt RUN_V3_SEMANTIC_AUDIT=True và chạy lại cell này.")

if SEMANTIC_THRESHOLD is not None:
    threshold = float(SEMANTIC_THRESHOLD)
    if not -1.0 <= threshold <= 1.0:
        raise ValueError("SEMANTIC_THRESHOLD phải nằm trong [-1, 1]")
    if not audit_report_path.is_file():
        raise FileNotFoundError("Chưa có semantic_audit.json; không được freeze trước audit")
    if V3_FROZEN_DIR.exists() and any(V3_FROZEN_DIR.iterdir()):
        raise FileExistsError(f"Frozen output đã tồn tại: {V3_FROZEN_DIR}")
    freeze_command = [sys.executable, "-u", "-m", "src.data.finalize",
        "--input-dir", str(V3_AUDIT_DIR), "--output-dir", str(V3_FROZEN_DIR),
        "--dataset-id", "legal_envi_clean_semantic_seed67_v3",
        "--split-seed", "67", "--minimum-alignment-score", "0.75",
        "--semantic-model", "sentence-transformers/LaBSE",
        "--semantic-model-revision", "836121a",
        "--minimum-semantic-similarity", str(threshold), "--reuse-semantic-scores"]
    subprocess.run(freeze_command, cwd=PROJECT_DIR, check=True)
    create_command = [sys.executable, "-u", "-m", "src.vinai_baseline.create_configs",
        "--project-root", str(PROJECT_DIR), "--dataset-dir", str(V3_FROZEN_DIR),
        "--output-dir", str(V3_CONFIG_DIR), "--learning-rate", "0.00005",
        "--checkpoint-samples", "256", "--length-penalty", "0.8",
        "--repetition-penalty", "1.1", "--max-new-tokens", "448"]
    subprocess.run(create_command, cwd=PROJECT_DIR, check=True)
    frozen_zip = shutil.make_archive("/kaggle/working/processed_clean_semantic_seed67_v3",
                                      "zip", root_dir=V3_FROZEN_DIR)
    print("Frozen v3:", V3_FROZEN_DIR)
    print("Config v3:", V3_CONFIG_DIR)
    print("ZIP frozen để lưu lại:", frozen_zip)

## 6. Chọn thí nghiệm và resume

Chọn `DATASET_PROFILE="v2"` để dùng ngay dataset baseline ổn định; notebook sẽ tự chọn config trong `configs/vinai`. Chỉ chọn `v3_frozen` sau khi semantic audit/freeze hoàn tất. Giữ `RUN_FULL=False` để smoke trước. Khi resume từ output của phiên Kaggle cũ, attach output đó làm Input, điền `RUN_ID_OVERRIDE`, đặt `RESUME_FROM_CHECKPOINT='last'`, `RUN_FULL=True` và trỏ `RESUME_RUN_SOURCE` tới đúng thư mục run trong `/kaggle/input`.

In [ ]:
import json, re
from datetime import datetime, timezone
DATASET_PROFILE = "v2"  # v2 | v3_frozen
DIRECTION = "en2vi"
MODE = "lora"  # zero_shot | lora | full
CONFIG_OVERRIDE = ""  # chỉ dùng cho config tùy chỉnh; bình thường để trống
RUN_FULL = False
MEMBER_TAG = "thuan"
EXPERIMENT_TAG = "baseline_v2_t4safe16_kaggle"
RUN_ID_OVERRIDE = ""
RESUME_FROM_CHECKPOINT = ""
RESUME_RUN_SOURCE = ""  # ví dụ /kaggle/input/previous-run/.../<RUN_ID>

if DATASET_PROFILE not in {"v2", "v3_frozen"}: raise ValueError("DATASET_PROFILE không hợp lệ")
if DIRECTION not in {"en2vi", "vi2en"}: raise ValueError("DIRECTION không hợp lệ")
if MODE not in {"zero_shot", "lora", "full"}: raise ValueError("MODE không hợp lệ")
for label, value in {"MEMBER_TAG": MEMBER_TAG, "EXPERIMENT_TAG": EXPERIMENT_TAG}.items():
    if not re.fullmatch(r"[a-z0-9_-]+", value): raise ValueError(f"{label} không hợp lệ")
if RUN_ID_OVERRIDE and not re.fullmatch(r"[a-z0-9_-]+", RUN_ID_OVERRIDE):
    raise ValueError("RUN_ID_OVERRIDE không hợp lệ")
if RESUME_FROM_CHECKPOINT and (not RUN_ID_OVERRIDE or not RUN_FULL or MODE == "zero_shot"):
    raise ValueError("Resume cần RUN_ID_OVERRIDE, RUN_FULL=True và mode lora/full")
default_config_dir = PROJECT_DIR / ("configs/vinai" if DATASET_PROFILE == "v2" else "configs/generated_v3")
CONFIG = PROJECT_DIR / CONFIG_OVERRIDE if CONFIG_OVERRIDE else default_config_dir / f"{DIRECTION}_{MODE}.json"
if not CONFIG.is_file():
    hint = ("Hãy dùng NLP_colab.zip có dataset v2." if DATASET_PROFILE == "v2" else
            "Hãy chạy semantic audit/freeze ở phần 5 để sinh config v3.")
    raise FileNotFoundError(f"Không tìm thấy config: {CONFIG}. {hint}")
config_preview = json.loads(CONFIG.read_text(encoding="utf-8"))
expected_training_mode = "none" if MODE == "zero_shot" else MODE
if config_preview["direction"] != DIRECTION or config_preview["training"]["mode"] != expected_training_mode:
    raise ValueError("DIRECTION/MODE không khớp với CONFIG_OVERRIDE")
DATASET_ID = config_preview["data"]["dataset_id"]
expected_dataset_id = {"v2": "legal_envi_clean_seed67_v2",
                       "v3_frozen": "legal_envi_clean_semantic_seed67_v3"}[DATASET_PROFILE]
if DATASET_ID != expected_dataset_id:
    raise ValueError(f"DATASET_PROFILE={DATASET_PROFILE} yêu cầu dataset_id={expected_dataset_id}, "
                     f"nhưng config đang dùng {DATASET_ID}")
TRAIN_SEED = config_preview["seed"]
DATA_DIR = PROJECT_DIR / Path(config_preview["data"]["manifest_file"]).parent
RUN_STAMP = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
RUN_ID = RUN_ID_OVERRIDE or f"{DATASET_ID}__{DIRECTION}__{MODE}__seed{TRAIN_SEED}__{MEMBER_TAG}__{EXPERIMENT_TAG}__{RUN_STAMP}"
OUTPUT_DIR = Path("/kaggle/working/NLP_runs") / DATASET_ID / DIRECTION / MODE / RUN_ID
if RESUME_FROM_CHECKPOINT:
    source = Path(RESUME_RUN_SOURCE)
    if not source.is_dir() or source.name != RUN_ID:
        raise FileNotFoundError("RESUME_RUN_SOURCE phải là đúng thư mục RUN_ID đã attach vào Kaggle Input")
    shutil.copytree(source, OUTPUT_DIR, dirs_exist_ok=True)
manifest_path = PROJECT_DIR / config_preview["data"]["manifest_file"]
if not manifest_path.is_file():
    raise FileNotFoundError(f"Thiếu manifest của {DATASET_PROFILE}: {manifest_path}")
manifest_preview = json.loads(manifest_path.read_text(encoding="utf-8"))
if manifest_preview.get("dataset_id") != DATASET_ID:
    raise ValueError("dataset_id trong config và manifest không khớp")
if manifest_preview.get("split_contract", {}).get("seed") != 67 or TRAIN_SEED != 67:
    raise ValueError("Split seed và training seed đều phải là 67")
if manifest_preview.get("semantic_audit_only"):
    raise ValueError("Semantic audit chỉ để duyệt score, không được dùng để train")
required = [DATA_DIR / f"segments_dev_{DIRECTION}.jsonl"]
if MODE != "zero_shot": required.append(DATA_DIR / f"segments_train_{DIRECTION}.jsonl")
missing = [str(p) for p in required if not p.exists()]
if missing: raise FileNotFoundError("Thiếu dữ liệu: " + ", ".join(missing))
training = config_preview["training"]
print("Dataset profile:", DATASET_PROFILE, "| dataset ID:", DATASET_ID)
print("Config:", CONFIG)
print("Run ID:", RUN_ID)
print("Output:", OUTPUT_DIR)
print("Profile: train batch", training.get("per_device_train_batch_size"),
      "x accumulation", training.get("gradient_accumulation_steps"),
      "| eval batch", training.get("per_device_eval_batch_size"),
      "| checkpoint samples", training.get("checkpoint_eval_samples"))

## 7. Audit và hàm chạy có log

Log được stream trực tiếp và lưu tại `/kaggle/working/vinai_cli_logs`. Preflight chặn ZIP cũ chưa có bản sửa LoRA.

In [ ]:
from time import perf_counter
def format_duration(seconds):
    total = round(seconds); hours, rem = divmod(total, 3600); minutes, seconds = divmod(rem, 60)
    return f"{hours:02d}:{minutes:02d}:{seconds:02d}"

def run_cli(command, *extra):
    args = [sys.executable, "-u", "-m", "src.vinai_baseline.cli", command,
            "--config", str(CONFIG), "--output-dir", str(OUTPUT_DIR), *map(str, extra)]
    started = perf_counter()
    print(f"\n[{datetime.now().astimezone().isoformat(timespec='seconds')}] BẮT ĐẦU")
    print("RUN:", " ".join(args), flush=True)
    env = os.environ.copy(); env["PYTHONUNBUFFERED"] = "1"
    log_dir = Path("/kaggle/working/vinai_cli_logs"); log_dir.mkdir(parents=True, exist_ok=True)
    log_path = log_dir / f"{command}_{datetime.now().strftime('%Y%m%dT%H%M%S%f')}.log"
    lines = []
    with log_path.open("w", encoding="utf-8") as log_file:
        process = subprocess.Popen(args, cwd=PROJECT_DIR, env=env, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, encoding="utf-8",
                                   errors="replace", bufsize=1)
        for line in process.stdout:
            print(line, end="", flush=True); log_file.write(line); log_file.flush(); lines.append(line.rstrip())
        return_code = process.wait()
    elapsed = perf_counter() - started
    if return_code:
        tail = "\n".join(lines[-80:]) or "(không có log)"
        raise RuntimeError(f"Lệnh {command} lỗi code {return_code} sau {format_duration(elapsed)}\n{tail}\nLog: {log_path}") from None
    print("HOÀN THÀNH sau", format_duration(elapsed), "| Log:", log_path)
    return elapsed

collator_source = (PROJECT_DIR / "src/vinai_baseline/data.py").read_text(encoding="utf-8")
if "prepare_decoder_input_ids_from_labels" not in collator_source:
    raise RuntimeError("ZIP cũ, chưa có bản sửa labels/logits LoRA")
print("Preflight LoRA collator: OK")
run_cli("audit")

## 8. Smoke test

LoRA/full dùng 32 mẫu train và 16 validation; zero-shot dùng 16 validation. Smoke chỉ xác minh pipeline, không kết luận chất lượng.

In [ ]:
COMMAND = "evaluate" if MODE == "zero_shot" else "train"
if RESUME_FROM_CHECKPOINT:
    RESULT_DIR = OUTPUT_DIR
    print("Bỏ qua smoke vì đang resume.")
else:
    SMOKE_OUTPUT = Path("/kaggle/working/vinai_smoke") / RUN_ID
    full_output = OUTPUT_DIR
    try:
        OUTPUT_DIR = SMOKE_OUTPUT
        if MODE == "zero_shot":
            run_cli(COMMAND, "--max-validation-samples", "16")
        else:
            run_cli(COMMAND, "--max-train-samples", "32", "--max-validation-samples", "16")
            run_cli("verify", "--max-validation-samples", "16")
        RESULT_DIR = SMOKE_OUTPUT
    finally:
        OUTPUT_DIR = full_output
    print("Smoke hoàn thành:", RESULT_DIR)

## 9. Full run

Sau khi smoke pass, đổi `START_FULL_NOW=True` ngay trong cell dưới rồi chạy cell này; không chạy lại cell cấu hình nên `RUN_ID` được giữ nguyên. Khi resume, `RUN_FULL=True` từ cell cấu hình cũng tự kích hoạt full run. Output nằm trong `/kaggle/working/NLP_runs`; hãy **Save Version** hoặc tải ZIP kết quả ở cell cuối trước khi phiên kết thúc.

In [ ]:
START_FULL_NOW = False  # đổi True sau khi smoke pass; chỉ chạy lại cell này
if START_FULL_NOW:
    RUN_FULL = True
if RUN_FULL:
    COMMAND = "evaluate" if MODE == "zero_shot" else "train"
    resume_args = (["--resume-from-checkpoint", RESUME_FROM_CHECKPOINT] if RESUME_FROM_CHECKPOINT else [])
    run_cli(COMMAND, *resume_args)
    if MODE != "zero_shot": run_cli("verify")
    RESULT_DIR = OUTPUT_DIR
else:
    print("Chưa chạy full; đang dùng kết quả smoke:", RESULT_DIR)

## 10. Xem metric, ví dụ dịch và đóng gói output

Cell kiểm tra đủ prediction, hiển thị metric/mẫu dịch và tạo một ZIP trong Kaggle Output để tải về hoặc dùng làm Input cho phiên resume.

In [ ]:
import random, pandas as pd
from IPython.display import display
RESULT_DIR = Path(RESULT_DIR)
metrics_path = RESULT_DIR / "reports/validation_metrics.json"
if not metrics_path.exists(): raise FileNotFoundError("Run chưa có validation_metrics.json")
metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
prediction_path = RESULT_DIR / "predictions/validation_predictions.jsonl"
predictions = [json.loads(line) for line in prediction_path.read_text(encoding="utf-8").splitlines() if line.strip()]
expected = int(metrics["evaluated_records"])
if len(predictions) != expected: raise RuntimeError(f"Prediction {len(predictions)}/{expected}")
def metric(name): return metrics.get(f"validation_{name}", metrics.get(name))
summary = [(name, metric(name)) for name in ["bleu", "chrf", "empty_output_rate",
           "length_ratio_mean", "length_ratio_p95", "length_ratio_over_3_rate",
           "repeated_trigram_rate", "dominant_token_over_20_rate", "number_preservation_f1",
           "source_number_preservation_f1", "reference_preservation_f1",
           "legal_identifier_preservation_f1", "source_legal_identifier_preservation_f1", "runtime"]]
summary.append(("peak_vram_mb", metrics.get("peak_vram_mb")))
display(pd.DataFrame(summary, columns=["metric", "value"]))
indices = random.Random(67).sample(range(len(predictions)), min(5, len(predictions)))
display(pd.DataFrame([{k: predictions[i][k] for k in ["segment_id", "source", "prediction", "target"]} for i in indices]))
archive_base = Path("/kaggle/working") / f"{RUN_ID}_result"
archive_path = shutil.make_archive(str(archive_base), "zip", root_dir=RESULT_DIR)
print("Artifact đầy đủ:", len(predictions), "prediction")
print("ZIP kết quả:", archive_path)

## 11. Generation sweep tùy chọn (không train lại)

Chỉ bật sau khi đã có `best_model`. Cell dùng cùng subset cố định để thử các cấu hình chống output dài/lặp và ghi từng bộ prediction riêng.

In [ ]:
RUN_GENERATION_SWEEP = False
if RUN_GENERATION_SWEEP:
    if MODE != "zero_shot" and Path(RESULT_DIR).resolve() != Path(OUTPUT_DIR).resolve():
        raise RuntimeError("Không sweep adapter smoke; hãy hoàn thành full run trước.")
    sweep_dir = Path("/kaggle/working/generation_sweeps") / f"{RUN_ID}__{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}"
    sweep_command = [sys.executable, "-u", "-m", "src.vinai_baseline.sweep_cli",
                     "--config", str(CONFIG), "--output-dir", str(sweep_dir),
                     "--samples", "256", "--batch-size", "16"]
    if MODE != "zero_shot":
        sweep_command.extend(["--saved-model-dir", str(Path(RESULT_DIR) / "best_model")])
    subprocess.run(sweep_command, cwd=PROJECT_DIR, check=True)
    print("Sweep summary:", sweep_dir / "sweep_summary.json")
else:
    print("Bỏ qua sweep; đổi RUN_GENERATION_SWEEP=True khi cần.")